In [11]:
import os
import json
import pdfplumber

print("Biblioteki załadowane")

Biblioteki załadowane


In [12]:
PDF_FILENAME = "ICh_kryteria-kwalifikacyjne-rekrutacji-na-II-stopien_c108abb2.pdf" 
PDF_PATH = os.path.join("data", PDF_FILENAME)

if os.path.exists(PDF_PATH):
    print(f"Znaleziono plik: {PDF_PATH}")
else:
    print(f"Nie znaleziono pliku: {PDF_PATH}")

Znaleziono plik: data\ICh_kryteria-kwalifikacyjne-rekrutacji-na-II-stopien_c108abb2.pdf


In [13]:
full_document_text = ""

print(f"Ekstrakcja tekstu z: {PDF_FILENAME}\n")

with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text(layout=True) 
        
        if text:
            full_document_text += text + "\n"
            print(f"Strona {i+1}: Wyciągnięto {len(text.split())} słów.")
        else:
            print(f"Strona {i+1}: [UWAGA] Brak tekstu. Możliwy skan (wymagany OCR).")

print("\nPodgląd pierwszych 300 znaków wyciągniętego tekstu:")
print("-" * 50)
print(full_document_text.strip()[:300] + "\n[...]")
print("-" * 50)

Ekstrakcja tekstu z: ICh_kryteria-kwalifikacyjne-rekrutacji-na-II-stopien_c108abb2.pdf

Strona 1: Wyciągnięto 305 słów.
Strona 2: Wyciągnięto 307 słów.
Strona 3: Wyciągnięto 313 słów.
Strona 4: Wyciągnięto 303 słów.
Strona 5: Wyciągnięto 185 słów.
Strona 6: Wyciągnięto 293 słów.
Strona 7: Wyciągnięto 248 słów.
Strona 8: Wyciągnięto 244 słów.
Strona 9: Wyciągnięto 230 słów.
Strona 10: Wyciągnięto 228 słów.
Strona 11: Wyciągnięto 269 słów.
Strona 12: Wyciągnięto 242 słów.
Strona 13: Wyciągnięto 271 słów.
Strona 14: Wyciągnięto 275 słów.
Strona 15: Wyciągnięto 238 słów.
Strona 16: Wyciągnięto 203 słów.
Strona 17: Wyciągnięto 177 słów.

Podgląd pierwszych 300 znaków wyciągniętego tekstu:
--------------------------------------------------
Szczegółowe kryteria kwalifikacyjne rekrutacji        
                              na I rok stacjonarnych studiów II stopnia           
                                                                                  
                             Kierun

In [14]:
def split_text_into_chunks(text, max_words=250, overlap_words=40):
    words = text.split()
    chunks = []
    
    if len(words) <= max_words:
        return [text]
        
    start = 0
    while start < len(words):
        end = start + max_words
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)
        chunks.append(chunk_text)
        
        start += (max_words - overlap_words)
        
    return chunks

document_chunks = split_text_into_chunks(full_document_text)

print(f"Dokument podzielono na {len(document_chunks)} fragmentów")

Dokument podzielono na 21 fragmentów


In [15]:
poc_jsonl_data = []

for chunk in document_chunks:
    '''Słowniki wg wytycznych zadania. Dla pliku wyjściowego'''
    json_object = {
        "text": chunk,
        "tokens": [],    
        "labels": [],    
        "entities": []   
    }
    poc_jsonl_data.append(json_object)

print("Struktura danych dla pierwszego fragmentu:")
print(json.dumps(poc_jsonl_data[0], indent=4, ensure_ascii=False)[:300] + "\n...")

Struktura danych dla pierwszego fragmentu:
{
    "text": "Szczegółowe kryteria kwalifikacyjne rekrutacji na I rok stacjonarnych studiów II stopnia Kierunek Inżynieria Chemiczna i procesowa 1. Tryb rekrutacji • Podstawę przyjęcia na studia stanowi uzyskanie: ✓ wskaźnika rekrutacyjnego o wartości wyższej bądź równej wartości minimalnej ustalon
...


In [16]:
output_file = "poc_output.jsonl"

with open(output_file, 'w', encoding='utf-8') as f:
    for item in poc_jsonl_data:
        # ensure_ascii=False gwarantuje, że polskie znaki (ą, ę, ł) nie zamienią się w "krzaczki"
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
print(f"Pomyślnie zapisano strukturę {len(poc_jsonl_data)} linii do pliku: {output_file}")

Pomyślnie zapisano strukturę 21 linii do pliku: poc_output.jsonl


In [ ]:
from transformers import AutoTokenizer

model_name = "allegro/herbert-base-cased"

print(f"pobieranie i ładowanie tokenizera: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer gotowy")

pobieranie i ładowanie tokenizera: allegro/herbert-base-cased...


config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

c:\Users\WiktorBurnecki\Desktop\NamedEntityRecognition\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WiktorBurnecki\.cache\huggingface\hub\models--allegro--herbert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/907k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/556k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Tokenizer gotowy


In [ ]:
#fragment testowy
sample_text = "Szczegółowe kryteria kwalifikacyjne rekrutacji na I rok stacjonarnych studiów II stopnia Kierunek Inżynieria Chemiczna."

encoded = tokenizer(sample_text, add_special_tokens=True)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

word_ids = encoded.word_ids()

print(f"Oryginalny tekst:\n{sample_text}\n")
print(f"{'Token (Subword)':<20} | {'Token ID':<10} | {'Word ID (Indeks Słowa)'}")
print("-" * 60)

for token, t_id, w_id in zip(tokens, encoded["input_ids"], word_ids):
    # Wyświetlamy token, jego ID oraz word_id
    print(f"{token:<20} | {t_id:<10} | {w_id}")